# Retinal Vessel Segmentation

**Group:**
* Jakub Biernat 160248
* Eryk Masian 160228

**Technologies Used:**
* **Language:** Python
* **Libraries:** TODO

## Imports

In [70]:
#General imports
from skimage import io
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import numpy as np

#Imports for Retinal Vessel Detection via Image Processing
from skimage.filters import threshold_otsu, gaussian, sobel
from skimage.morphology import opening, closing
from skimage.color import rgb2gray
from skimage import exposure

## Images
From HRF image database: https://www5.cs.fau.de/research/data/fundus-images/

In [71]:
image_names = [f"{str(i).zfill(2)}_{suffix}" for i in range(1, 16) for suffix in ["h", "g", "dr"]]

def load_images(image_name):
    raw_image = io.imread(f"../data/images/{image_name}.jpg")
    gs_image = io.imread(f"../data/goldstandard/{image_name}.tif")
    mask = io.imread(f"../data/fovs/{image_name}_mask.tif")
    return raw_image, gs_image, mask


## Quality metrics and visualisation

In [72]:
def generate_gs_comparison(gs_image, v_image):
    gs = gs_image > 0
    pred = v_image > 0

    comparison_image = np.zeros((*v_image.shape, 3), dtype=np.uint8)

    #True Positive
    tp = gs & pred
    comparison_image[tp] = [255, 255, 255]

    #False Positive
    fp = pred & ~gs
    comparison_image[fp] = [255, 0, 0]

    #False Negative
    fn = gs & ~pred
    comparison_image[fn] = [0, 0, 255]

    return comparison_image

def generate_overlay(raw_image, v_image):
    pred = v_image > 0

    overlay = raw_image.copy()
    overlay[pred] = [0, 255, 0]

    return overlay

#TODO: Quality Metrics

## Retinal Vessel Detection via Image Processing

### Image processing

In [73]:
def retinal_vessel_segmentation_image_processing(image, mask):
    ######## Pre-processing ########
    image = rgb2gray(image)

    image = gaussian(image, sigma=1)

    image = exposure.equalize_hist(image)

    ######## Core processing ########
    image = sobel(image)

    ######## Post-processing ########
    image = image > threshold_otsu(image)

    image = closing(image)
    image = opening(image)

    return image

### App

In [74]:
# ---------------- UI ----------------
image_selector = widgets.Dropdown(
    options=image_names,
    description="Obraz:"
)

segment_button = widgets.Button(
    description="Segmentacja",
    button_style="success",
    icon="play"
)

preview_output = widgets.Output()
result_output = widgets.Output()

# ---------------- PREVIEW ----------------
def show_preview(change=None):

    with preview_output:
        clear_output(wait=True)

        image_name = image_selector.value
        raw_image, _, _ = load_images(image_name)

        plt.figure(figsize=(5, 5))
        plt.imshow(raw_image, cmap="gray")
        plt.title(image_name)
        plt.axis("off")
        plt.show()

# ---------------- SEGMENTATION ----------------
def run_segmentation(button):

    with result_output:
        clear_output(wait=True)

        image_name = image_selector.value

        raw_image, gs_image, mask = load_images(image_name)

        vessel_image = retinal_vessel_segmentation_image_processing(raw_image, mask)
        comparison_image = generate_gs_comparison(gs_image, vessel_image)
        overlay = generate_overlay(raw_image, vessel_image)

        fig, axes = plt.subplots(2, 2, figsize=(15, 5))

        axes[0][0].imshow(vessel_image, cmap="gray")
        axes[0][0].set_title("Wygenerowana segmentacja")
        axes[0][0].axis("off")

        axes[0][1].imshow(gs_image, cmap="gray")
        axes[0][1].set_title("Gold standard")
        axes[0][1].axis("off")

        axes[1][0].imshow(comparison_image, cmap="gray")
        axes[1][0].set_title("Porównanie wygenerowanej segmentacji oraz standardu")
        axes[1][0].axis("off")

        axes[1][1].imshow(overlay, cmap="gray")
        axes[1][1].set_title("Wykryte naczynia na obrazie wejściowym")
        axes[1][1].axis("off")

        plt.tight_layout()
        plt.show()

# ---------------- OBSERVERS ----------------
image_selector.observe(show_preview, names="value")
segment_button.on_click(run_segmentation)

# ---------------- LAYOUT ----------------
left_panel = widgets.VBox([
    image_selector,
    segment_button
])

top_panel = widgets.HBox([
    left_panel,
    preview_output
])

display(
    widgets.VBox([
        top_panel,
        result_output
    ])
)

# pierwszy podgląd
show_preview()